# 🔬 RoomBeacon — Exploratory Data Analysis & Feature Engineering

**Mục tiêu:** Khám phá, đánh giá chất lượng, làm sạch và trích xuất đặc trưng
từ dữ liệu cho thuê phòng trọ/nhà ở Việt Nam thu thập qua hệ thống RoomBeacon.

**Phạm vi dữ liệu:**
- Nguồn: `v_latest_posts` mở rộng (MySQL Bronze → DuckDB READ_ONLY)
- Đơn vị phân tích: Mỗi dòng = 1 listing duy nhất (latest observation)
- Các nguồn web: PhongTro123, NhaTroVN, NhaTot, BatDongSan, ChoThueNha, TroMoi, CafeLand...

**Nguyên tắc pipeline:**
```
df_raw (immutable) → df_work (profiling & cleaning) → df_clean (validated) → export
```

**Quy tắc:**
- `df_raw` KHÔNG BAO GIỜ bị sửa đổi — dùng để audit và so sánh before/after
- Cleaning = flag & replace, KHÔNG drop rows bừa bãi
- Mọi hàm logic nặng nằm trong `utils/` — notebook chỉ gọi và hiển thị

## 1. Imports & Config

In [ ]:
import warnings
warnings.filterwarnings('ignore')

# --- Project path setup (giữ nguyên) ---
try:
    from utils import setup_project_path
except ModuleNotFoundError:
    from notebooks.utils import setup_project_path

PROJECT_ROOT = setup_project_path()
from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / ".env", override=False)

# --- Core libraries ---
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from datetime import datetime, timezone
from IPython.display import display, Markdown

# --- DuckDB connection (giữ nguyên) ---
from analytics.duckdb.connection import create_analytics_connection

# --- Utility modules ---
from utils.address_cleaner import clean_address
from utils.location_normalizer import normalize_location
from utils.price_validator import (
    validate_price, classify_price_band,
    detect_price_outliers_iqr, get_price_summary,
)
from utils.area_validator import (
    validate_area, classify_area_band, detect_area_outliers_iqr,
)
from utils.data_quality import (
    compute_missing_profile, compute_coverage_by_source,
    compute_quality_score, generate_quality_report, detect_duplicates,
)

In [ ]:
# --- Plot config ---
plt.style.use('ggplot')
sns.set_theme(style="whitegrid", rc={"axes.facecolor": "#F9F9F9"})
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 11
pd.set_option('display.max_columns', 25)
pd.set_option('display.max_colwidth', 60)

## 2. Load Data

Kết nối DuckDB (READ_ONLY attach MySQL Bronze) và load dữ liệu mở rộng.

In [ ]:
# Load extended EDA query
eda_sql_path = PROJECT_ROOT / "notebooks" / "sql" / "eda_latest_posts.sql"
EDA_QUERY = eda_sql_path.read_text(encoding="utf-8")

# Connect & execute
conn = create_analytics_connection()
df_raw = conn.sql(EDA_QUERY).df()

print(f"✓ Loaded {df_raw.shape[0]:,} listings × {df_raw.shape[1]} columns")

Kiểm tra kích thước và xem qua dữ liệu thô.

In [ ]:
print(f"Shape: {df_raw.shape}")
print(f"Columns: {list(df_raw.columns)}")
display(df_raw.head(3))

## 3. Dataset Overview

**Câu hỏi:** Cấu trúc dữ liệu gồm bao nhiêu cột, kiểu dữ liệu nào?

In [ ]:
print(f"Kích thước: {df_raw.shape[0]:,} dòng × {df_raw.shape[1]} cột")
print(f"\nKiểu dữ liệu:")
display(df_raw.dtypes.to_frame('dtype').reset_index().rename(columns={'index': 'column'}))

**Câu hỏi:** Dữ liệu thô trông như thế nào?

In [ ]:
display(df_raw.sample(5, random_state=42))

**Câu hỏi:** Phân bố listings theo nguồn website?

In [ ]:
source_dist = df_raw['source'].value_counts()
print(f"Số nguồn: {source_dist.shape[0]}")
print(f"Unique listings: {df_raw['rental_post_id'].nunique():,}")
display(source_dist.to_frame('count'))

fig, ax = plt.subplots(figsize=(10, 4))
source_dist.plot.barh(ax=ax, color=sns.color_palette('Set2', len(source_dist)))
ax.set_title('Số lượng Listings theo Nguồn Website', fontweight='bold')
ax.set_xlabel('Số listings')
for i, v in enumerate(source_dist.values):
    ax.text(v + 10, i, f'{v:,}', va='center', fontsize=10)
plt.tight_layout()
plt.show()

**Kết luận:** Xem biểu đồ trên để hiểu tỷ trọng đóng góp từ mỗi nguồn.

## 4. Create Working DataFrame

Tạo bản sao `df_work` để thao tác. **`df_raw` giữ nguyên** cho audit và so sánh before/after.

In [ ]:
df_work = df_raw.copy()
print(f"df_raw  shape: {df_raw.shape}")
print(f"df_work shape: {df_work.shape}")
print(f"\n→ df_raw sẽ KHÔNG bị sửa đổi từ đây trở đi.")

## 5. Data Quality Profiling

### 5.1 Missing Values

**Câu hỏi:** Tỷ lệ dữ liệu khuyết trên từng trường?

In [ ]:
missing_profile = compute_missing_profile(df_work)
display(missing_profile[missing_profile['missing_count'] > 0])

# Visualization
missing_cols = missing_profile[missing_profile['missing_count'] > 0]
if not missing_cols.empty:
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    # Bar chart
    sns.barplot(data=missing_cols, x='missing_rate_pct', y='column',
               palette='Reds_r', ax=axes[0])
    axes[0].set_title('Tỷ lệ Dữ liệu Khuyết (%)', fontweight='bold')
    axes[0].set_xlim(0, 100)

    # Heatmap
    sns.heatmap(df_work.isnull(), cbar=False, cmap='viridis',
               yticklabels=False, ax=axes[1])
    axes[1].set_title('Missing Pattern Heatmap', fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print('Không có cột nào bị khuyết dữ liệu!')

**Kết luận:** Xem bảng và biểu đồ trên để xác định các trường cần xử lý.

### 5.2 Duplicate Records

**Câu hỏi:** Có bản ghi trùng lặp theo `rental_post_id`?

In [ ]:
dupes = detect_duplicates(df_work, ['rental_post_id'])
print(f"Số bản ghi trùng: {len(dupes):,}")
if not dupes.empty:
    display(dupes.head(10))
else:
    print("✓ Không có duplicate — mỗi rental_post_id là duy nhất.")

### 5.3 Coverage by Source

**Câu hỏi:** Mỗi nguồn cung cấp đầy đủ price/area/address ở mức nào?

In [ ]:
key_cols = ['price', 'area', 'address', 'title_raw', 'latitude']
coverage = compute_coverage_by_source(df_work, key_cols)
display(coverage)

# Heatmap visualization
pct_cols = [c for c in coverage.columns if c.endswith('_pct')]
if pct_cols:
    heatmap_data = coverage.set_index('source')[pct_cols]
    heatmap_data.columns = [c.replace('_pct', '') for c in pct_cols]

    plt.figure(figsize=(10, max(3, len(heatmap_data) * 0.5)))
    sns.heatmap(heatmap_data, annot=True, fmt='.1f', cmap='RdYlGn',
               vmin=0, vmax=100, linewidths=1)
    plt.title('Coverage (%) theo Source', fontweight='bold')
    plt.ylabel('')
    plt.tight_layout()
    plt.show()

**Kết luận:** Coverage heatmap cho thấy nguồn nào mạnh/yếu ở trường nào.

### 5.4 Invalid / Suspicious Values

**Câu hỏi:** Có giá trị bất hợp lý (giá ≤ 0, diện tích ≤ 0, giá quá lớn)?

In [ ]:
invalid_checks = {
    'price ≤ 0': df_work['price'].notna() & (df_work['price'] <= 0),
    'price > 200M': df_work['price'].notna() & (df_work['price'] > 200_000_000),
    'area ≤ 0': df_work['area'].notna() & (df_work['area'] <= 0),
    'area > 1000 m²': df_work['area'].notna() & (df_work['area'] > 1000),
    'address empty string': df_work['address'].notna() & (df_work['address'].str.strip() == ''),
}

invalid_summary = pd.DataFrame([
    {'check': k, 'count': int(v.sum())}
    for k, v in invalid_checks.items()
])
display(invalid_summary)

### 5.5 Data Quality Summary

In [ ]:
total = len(df_work)
summary = {
    'Total listings': f'{total:,}',
    'Unique rental_post_id': f'{df_work["rental_post_id"].nunique():,}',
    'Sources': df_work['source'].nunique(),
    'Price coverage': f"{df_work['price'].notna().sum():,} ({df_work['price'].notna().mean()*100:.1f}%)",
    'Area coverage': f"{df_work['area'].notna().sum():,} ({df_work['area'].notna().mean()*100:.1f}%)",
    'Address coverage': f"{df_work['address'].notna().sum():,} ({df_work['address'].notna().mean()*100:.1f}%)",
    'Lat/Lon coverage': f"{df_work['latitude'].notna().sum():,} ({df_work['latitude'].notna().mean()*100:.1f}%)",
}
display(pd.DataFrame(summary.items(), columns=['Metric', 'Value']))

## 6. Price EDA & Cleaning

### 6.1 Missing Price

**Câu hỏi:** Bao nhiêu listings thiếu giá?

In [ ]:
price_missing = df_work['price'].isna().sum()
price_present = df_work['price'].notna().sum()
print(f"Có giá:    {price_present:,} ({price_present/len(df_work)*100:.1f}%)")
print(f"Thiếu giá: {price_missing:,} ({price_missing/len(df_work)*100:.1f}%)")

### 6.2 Descriptive Statistics

**Câu hỏi:** Phân bố giá ra sao?

In [ ]:
price_stats = get_price_summary(df_work['price'])
display(pd.DataFrame(price_stats.items(), columns=['Statistic', 'Value']))

### 6.3 Distribution

**Câu hỏi:** Phân phối giá thuê có bị lệch (skewed)?

In [ ]:
price_valid = df_work['price'].dropna()
price_million = price_valid / 1_000_000

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Histogram
sns.histplot(price_million, bins=80, ax=axes[0], color='#3498db', kde=True)
axes[0].set_title('Phân phối Giá thuê (Triệu VND)', fontweight='bold')
axes[0].set_xlabel('Giá thuê (Triệu VND)')
axes[0].set_xlim(0, price_million.quantile(0.98))

# Boxplot
sns.boxplot(x=price_million, ax=axes[1], color='#3498db')
axes[1].set_title('Boxplot Giá thuê', fontweight='bold')
axes[1].set_xlabel('Giá thuê (Triệu VND)')
axes[1].set_xlim(0, price_million.quantile(0.98))

plt.tight_layout()
plt.show()

**Kết luận:** Phân phối giá thuê thường lệch phải (right-skewed) — phần lớn phòng trọ có giá thấp.

### 6.4 Outlier Detection

**Câu hỏi:** Có giá trị ngoại lệ nào?

In [ ]:
price_for_iqr = df_work.loc[df_work['price'].notna(), 'price'] / 1_000_000
outlier_mask = detect_price_outliers_iqr(price_for_iqr)
n_outliers = outlier_mask.sum()
print(f"Outliers (IQR): {n_outliers:,} / {len(price_for_iqr):,} ({n_outliers/len(price_for_iqr)*100:.1f}%)")

if n_outliers > 0:
    print(f"\nOutlier range: < {price_for_iqr[~outlier_mask].min():.2f} triệu hoặc > {price_for_iqr[~outlier_mask].max():.2f} triệu")
    print(f"\nTop 5 giá cao nhất:")
    display(df_work.loc[price_for_iqr.nlargest(5).index, ['source', 'listing_id', 'price', 'area', 'address']])

### 6.5 Inspect Suspicious Records

**Câu hỏi:** Các bản ghi có giá bất thường trông như thế nào?

In [ ]:
# Records ngoài plausibility range
suspicious_price = df_work[
    df_work['price'].notna() & ~df_work['price'].apply(validate_price)
]
print(f"Records ngoài plausibility range (300K – 200M VND): {len(suspicious_price):,}")
if not suspicious_price.empty:
    display(suspicious_price[['source', 'listing_id', 'price', 'area', 'title_raw']].head(10))

### 6.6 Apply Price Rules

Áp dụng plausibility guard:
- Giá hợp lệ: 300K ≤ price ≤ 200M VND
- Giá ngoài khoảng → đánh dấu `price_valid = False`, giữ `price` gốc để audit

In [ ]:
df_work['price_valid'] = df_work['price'].apply(validate_price)
df_work['price_cleaned'] = df_work['price'].where(df_work['price_valid'])
df_work['price_million'] = df_work['price_cleaned'] / 1_000_000

n_invalid = (~df_work['price_valid']).sum()
n_valid = df_work['price_valid'].sum()
print(f"Price valid:   {n_valid:,}")
print(f"Price invalid: {n_invalid:,} (set to NaN in price_cleaned)")

### 6.7 Validate Cleaned Price

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# BEFORE
before = df_work['price'].dropna() / 1_000_000
sns.histplot(before, bins=80, ax=axes[0], color='#e74c3c', kde=True)
axes[0].set_title('Giá thuê — TRƯỚC cleaning', fontweight='bold')
axes[0].set_xlim(0, before.quantile(0.98))
axes[0].set_xlabel('Triệu VND')

# AFTER
after = df_work['price_million'].dropna()
sns.histplot(after, bins=80, ax=axes[1], color='#27ae60', kde=True)
axes[1].set_title('Giá thuê — SAU cleaning', fontweight='bold')
axes[1].set_xlim(0, after.quantile(0.98))
axes[1].set_xlabel('Triệu VND')

plt.tight_layout()
plt.show()
print(f"Trước: {len(before):,} records | Sau: {len(after):,} records")

**Kết luận:** Price cleaning loại bỏ các giá bất hợp lý, giữ nguyên row count.

## 7. Area EDA & Cleaning

### 7.1 Missing Area

**Câu hỏi:** Bao nhiêu listings thiếu diện tích?

In [ ]:
area_missing = df_work['area'].isna().sum()
area_present = df_work['area'].notna().sum()
print(f"Có diện tích:    {area_present:,} ({area_present/len(df_work)*100:.1f}%)")
print(f"Thiếu diện tích: {area_missing:,} ({area_missing/len(df_work)*100:.1f}%)")

### 7.2 Distribution

**Câu hỏi:** Phân phối diện tích cho thuê?

In [ ]:
area_valid = df_work['area'].dropna()

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

sns.histplot(area_valid, bins=80, ax=axes[0], color='#9b59b6', kde=True)
axes[0].set_title('Phân phối Diện tích (m²)', fontweight='bold')
axes[0].set_xlabel('Diện tích (m²)')
axes[0].set_xlim(0, area_valid.quantile(0.98))

sns.boxplot(x=area_valid, ax=axes[1], color='#9b59b6')
axes[1].set_title('Boxplot Diện tích', fontweight='bold')
axes[1].set_xlim(0, area_valid.quantile(0.98))

plt.tight_layout()
plt.show()

### 7.3 Invalid / Zero / Negative

**Câu hỏi:** Có diện tích ≤ 0 hoặc quá lớn?

In [ ]:
area_invalid = df_work[df_work['area'].notna() & ~df_work['area'].apply(validate_area)]
print(f"Diện tích ngoài khoảng hợp lệ (5 – 1000 m²): {len(area_invalid):,}")
if not area_invalid.empty:
    display(area_invalid[['source', 'listing_id', 'area', 'price', 'title_raw']].head(10))

### 7.4 Outliers

In [ ]:
area_for_iqr = df_work.loc[df_work['area'].notna(), 'area']
area_outlier_mask = detect_area_outliers_iqr(area_for_iqr)
n_area_outliers = area_outlier_mask.sum()
print(f"Area outliers (IQR): {n_area_outliers:,} / {len(area_for_iqr):,}")

### 7.5 Apply & Validate Area Rules

In [ ]:
df_work['area_valid'] = df_work['area'].apply(validate_area)
df_work['area_cleaned'] = df_work['area'].where(df_work['area_valid'])

n_area_inv = (~df_work['area_valid']).sum()
n_area_val = df_work['area_valid'].sum()
print(f"Area valid:   {n_area_val:,}")
print(f"Area invalid: {n_area_inv:,}")

# Before/After
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

sns.histplot(df_work['area'].dropna(), bins=80, ax=axes[0], color='#e74c3c', kde=True)
axes[0].set_title('Diện tích — TRƯỚC cleaning', fontweight='bold')
axes[0].set_xlim(0, df_work['area'].dropna().quantile(0.98))

sns.histplot(df_work['area_cleaned'].dropna(), bins=80, ax=axes[1], color='#27ae60', kde=True)
axes[1].set_title('Diện tích — SAU cleaning', fontweight='bold')
axes[1].set_xlim(0, df_work['area_cleaned'].dropna().quantile(0.98))

plt.tight_layout()
plt.show()

**Kết luận:** Area cleaning giữ nguyên row count, chỉ NaN hóa các giá trị bất hợp lý.

## 8. Location EDA & Normalization

### 8.1 Coverage of `address`

**Câu hỏi:** Bao nhiêu listings có địa chỉ?

In [ ]:
addr_present = df_work['address'].notna().sum()
addr_missing = df_work['address'].isna().sum()
print(f"Có address:    {addr_present:,} ({addr_present/len(df_work)*100:.1f}%)")
print(f"Thiếu address: {addr_missing:,} ({addr_missing/len(df_work)*100:.1f}%)")

### 8.2 Inherited Address

**Câu hỏi:** Bao nhiêu address được kế thừa từ version trước?

In [ ]:
if 'full_address_inherited' in df_work.columns:
    inherited = df_work['full_address_inherited'].sum()
    print(f"Inherited address: {inherited:,} / {addr_present:,} ({inherited/addr_present*100:.1f}%)")
else:
    print("Cột full_address_inherited không có trong dataset.")

### 8.3 Missing Address by Source

In [ ]:
addr_by_source = df_work.groupby('source').agg(
    total=('address', 'size'),
    has_address=('address', lambda x: x.notna().sum()),
).reset_index()
addr_by_source['coverage_pct'] = (addr_by_source['has_address'] / addr_by_source['total'] * 100).round(1)
display(addr_by_source.sort_values('coverage_pct'))

### 8.4 Sample Addresses

**Câu hỏi:** Địa chỉ thô trông như thế nào?

In [ ]:
sample_addrs = df_work[df_work['address'].notna()].groupby('source').head(2)
display(sample_addrs[['source', 'address']].head(12))

### 8.5 Clean Address

Chuẩn hóa định dạng địa chỉ (viết tắt, ký tự đặc biệt) giữ nguyên thông tin vị trí.

In [ ]:
df_work['address_clean'] = df_work['address'].apply(clean_address)

# Show examples
sample = df_work[df_work['address'].notna()].head(5)[['address', 'address_clean']]
display(sample)

### 8.6 Normalize Location

Ánh xạ phường cũ sang phường hiện hành (theo Nghị quyết sáp nhập đơn vị hành chính).

In [ ]:
def _primary_location(text):
    result = normalize_location(text)
    return result[0] if result else None

df_work['current_location'] = df_work['address_clean'].apply(_primary_location)

loc_mapped = df_work['current_location'].notna().sum()
loc_total = df_work['address_clean'].notna().sum()
print(f"Mapped location: {loc_mapped:,} / {loc_total:,} ({loc_mapped/loc_total*100:.1f}%)")

### 8.7 Location Distribution

**Câu hỏi:** Top phường/quận nào có nhiều listings nhất?

In [ ]:
loc_dist = df_work['current_location'].value_counts().head(20)

fig, ax = plt.subplots(figsize=(10, 8))
loc_dist.plot.barh(ax=ax, color=sns.color_palette('viridis', len(loc_dist)))
ax.set_title('Top 20 Phường có nhiều Listings nhất', fontweight='bold')
ax.set_xlabel('Số listings')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

### 8.8 Validate Mapping Results

In [ ]:
# Summary
print(f"Unique locations mapped: {df_work['current_location'].nunique()}")
print(f"Coverage: {df_work['current_location'].notna().mean()*100:.1f}%")
print(f"\nUnmapped addresses (sample):")
unmapped = df_work[
    df_work['address_clean'].notna() & df_work['current_location'].isna()
]
if not unmapped.empty:
    display(unmapped[['source', 'address', 'address_clean']].sample(min(10, len(unmapped)), random_state=42))
else:
    print("✓ Tất cả địa chỉ đã được ánh xạ thành công.")

## 9. Cross Analysis

### 9.1 Price × Area

**Câu hỏi:** Mối quan hệ giữa giá và diện tích?

In [ ]:
df_cross = df_work[df_work['price_million'].notna() & df_work['area_cleaned'].notna()].copy()
df_cross['price_per_m2'] = df_cross['price_million'] / df_cross['area_cleaned']

g = sns.jointplot(
    data=df_cross,
    x='area_cleaned', y='price_million',
    kind='hex', color='#2980b9', height=8, ratio=4
)
g.fig.suptitle('Mật độ 2D: Diện tích vs Giá thuê', y=1.03, fontweight='bold')
g.set_axis_labels('Diện tích (m²)', 'Giá thuê (Triệu VND)')
plt.show()

### 9.2 Price × Source

**Câu hỏi:** Giá thuê phân bố khác nhau theo nguồn website?

In [ ]:
plt.figure(figsize=(14, 6))
sns.violinplot(data=df_cross, x='source', y='price_million',
              palette='Set2', inner='quartile')
plt.title('Phân bố Giá thuê theo Nguồn Website', fontweight='bold')
plt.xlabel('Nguồn Website')
plt.ylabel('Giá thuê (Triệu VND)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

### 9.3 Area × Source

In [ ]:
plt.figure(figsize=(14, 6))
sns.violinplot(data=df_cross, x='source', y='area_cleaned',
              palette='Set3', inner='quartile')
plt.title('Phân bố Diện tích theo Nguồn Website', fontweight='bold')
plt.xlabel('Nguồn Website')
plt.ylabel('Diện tích (m²)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

### 9.4 Correlation Matrix

**Câu hỏi:** Tương quan giữa các biến số?

In [ ]:
numeric_cols = ['price_million', 'area_cleaned', 'price_per_m2']
if 'active_days' in df_cross.columns:
    numeric_cols.append('active_days')

corr = df_cross[numeric_cols].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f',
           linewidths=1, vmin=-1, vmax=1, square=True)
plt.title('Ma trận Tương quan (Pearson)', fontweight='bold')
plt.tight_layout()
plt.show()

**Kết luận:** Correlation matrix cho thấy mức độ liên hệ tuyến tính giữa các biến.

## 10. Feature Engineering

### 10.1 Price Features

- `price_million`: Giá quy triệu VND (đã tạo ở section 6)
- `price_per_m2`: Đơn giá mỗi m²
- `price_band`: Phân loại giá theo nhóm

In [ ]:
# price_million đã có từ section 6
df_work['price_per_m2'] = np.where(
    df_work['area_cleaned'].notna() & (df_work['area_cleaned'] > 0),
    df_work['price_million'] / df_work['area_cleaned'],
    np.nan
)
df_work['price_band'] = df_work['price_million'].apply(classify_price_band)

display(df_work['price_band'].value_counts().to_frame('count'))

### 10.2 Area Features

- `area_band`: Phân loại diện tích theo nhóm

In [ ]:
df_work['area_band'] = df_work['area_cleaned'].apply(classify_area_band)

display(df_work['area_band'].value_counts().to_frame('count'))

### 10.3 Location Features

- `current_location`: Phường hiện hành (đã tạo ở section 8)
- `has_coordinates`: Có tọa độ GPS?

In [ ]:
# current_location đã có từ section 8
df_work['has_coordinates'] = df_work['latitude'].notna() & df_work['longitude'].notna()

print(f"Listings có tọa độ GPS: {df_work['has_coordinates'].sum():,} / {len(df_work):,}")

### 10.4 Lifecycle Features

- `listing_age_days`: Số ngày từ lần đầu thấy đến nay
- `is_new_listing`: Listing mới (≤ 3 ngày)
- `days_since_last_seen`: Số ngày kể từ lần cuối crawl

In [ ]:
now = pd.Timestamp.now(tz='UTC')

if 'first_observed_at' in df_work.columns and df_work['first_observed_at'].notna().any():
    first_obs = pd.to_datetime(df_work['first_observed_at'], utc=True)
    last_obs = pd.to_datetime(df_work['last_observed_at'], utc=True)
    observed = pd.to_datetime(df_work['observed_at'], utc=True)

    df_work['listing_age_days'] = (now - first_obs).dt.days
    df_work['is_new_listing'] = df_work['listing_age_days'] <= 3
    df_work['days_since_last_seen'] = (now - last_obs).dt.days

    print(f"Listing mới (≤ 3 ngày): {df_work['is_new_listing'].sum():,}")
    print(f"Tuổi trung bình: {df_work['listing_age_days'].mean():.1f} ngày")
elif 'active_days' in df_work.columns:
    df_work['listing_age_days'] = df_work['active_days']
    df_work['is_new_listing'] = df_work['active_days'] <= 3
    df_work['days_since_last_seen'] = np.nan
    print(f"Sử dụng cột active_days. Listing mới: {df_work['is_new_listing'].sum():,}")
else:
    print("⚠ Không có cột lifecycle — bỏ qua.")
    df_work['listing_age_days'] = np.nan
    df_work['is_new_listing'] = np.nan
    df_work['days_since_last_seen'] = np.nan

### 10.5 Data Quality Features

- `has_price`, `has_area`, `has_address`: Boolean flags
- `data_quality_score`: 0.0 – 1.0

In [ ]:
df_work['has_price'] = df_work['price_cleaned'].notna()
df_work['has_area'] = df_work['area_cleaned'].notna()
df_work['has_address'] = df_work['address'].notna()
df_work['data_quality_score'] = df_work.apply(
    lambda r: compute_quality_score(r, ['price_cleaned', 'area_cleaned', 'address', 'title_raw']),
    axis=1
)

print(f"Data quality score distribution:")
display(df_work['data_quality_score'].value_counts().sort_index().to_frame('count'))

## 11. Feature Validation

**Câu hỏi:** Các features mới có giá trị bất thường (NaN, Inf, impossible values)?

In [ ]:
feature_cols = [
    'price_million', 'price_per_m2', 'price_band',
    'area_cleaned', 'area_band',
    'current_location', 'listing_age_days',
    'data_quality_score',
]
existing_features = [c for c in feature_cols if c in df_work.columns]

# Null / NaN check
null_check = df_work[existing_features].isnull().sum()
print("Null/NaN count per feature:")
display(null_check.to_frame('nulls'))

# Inf check (numeric only)
numeric_features = df_work[existing_features].select_dtypes(include=[np.number]).columns
inf_check = {col: np.isinf(df_work[col]).sum() for col in numeric_features if df_work[col].notna().any()}
print(f"\nInf values: {inf_check}")

**Phân phối features theo source:**

In [ ]:
if 'price_million' in df_work.columns:
    validation = df_work.groupby('source').agg(
        n=('source', 'size'),
        price_median=('price_million', 'median'),
        area_median=('area_cleaned', 'median'),
        quality_mean=('data_quality_score', 'mean'),
    ).round(2)
    display(validation)

## 12. Build Clean Dataset

Chọn final columns: cleaned values + raw audit columns + computed features.

In [ ]:
clean_columns = [
    # Identity
    'source', 'rental_post_id', 'listing_id',
    # Raw (audit)
    'title_raw', 'url', 'price', 'area', 'address',
    # Cleaned
    'price_cleaned', 'price_million', 'price_valid',
    'area_cleaned', 'area_valid',
    'address_clean', 'current_location',
    # Features
    'price_per_m2', 'price_band', 'area_band',
    'has_price', 'has_area', 'has_address', 'has_coordinates',
    'data_quality_score',
]

# Add lifecycle columns if available
lifecycle_cols = ['listing_age_days', 'is_new_listing', 'days_since_last_seen',
                  'first_observed_at', 'last_observed_at', 'observed_at', 'active_days']
for col in lifecycle_cols:
    if col in df_work.columns:
        clean_columns.append(col)

# Add coordinate columns if available
for col in ['latitude', 'longitude']:
    if col in df_work.columns:
        clean_columns.append(col)

# Filter to existing columns
final_columns = [c for c in clean_columns if c in df_work.columns]
df_clean = df_work[final_columns].copy()

print(f"df_clean shape: {df_clean.shape}")
print(f"Columns: {list(df_clean.columns)}")
display(df_clean.head(3))

**Kiểm tra:** `df_raw` vẫn nguyên vẹn?

In [ ]:
assert len(df_raw) == len(df_clean), "Row count changed! Check pipeline."
assert 'price_million' not in df_raw.columns, "df_raw was mutated!"
print(f"✓ df_raw vẫn nguyên: {df_raw.shape}")
print(f"✓ df_clean:          {df_clean.shape}")

## 13. Final Data Quality Report

So sánh coverage Before (raw) vs After (cleaned).

In [ ]:
report_cols = ['price', 'area', 'address']

# Map cleaned column names to raw names for comparison
df_after_for_report = df_clean.rename(columns={
    'price_cleaned': 'price',
    'area_cleaned': 'area',
})

report = generate_quality_report(df_raw, df_after_for_report, report_cols)
display(report)

# Visualization
fig, ax = plt.subplots(figsize=(10, 4))
x = np.arange(len(report))
width = 0.35
ax.bar(x - width/2, report['before_pct'], width, label='Before', color='#e74c3c', alpha=0.8)
ax.bar(x + width/2, report['after_pct'], width, label='After', color='#27ae60', alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels(report['column'])
ax.set_ylabel('Coverage %')
ax.set_ylim(0, 105)
ax.set_title('Data Coverage: Before vs After Cleaning', fontweight='bold')
ax.legend()
for i, (b, a) in enumerate(zip(report['before_pct'], report['after_pct'])):
    ax.text(i - width/2, b + 1, f'{b:.1f}%', ha='center', fontsize=9)
    ax.text(i + width/2, a + 1, f'{a:.1f}%', ha='center', fontsize=9)
plt.tight_layout()
plt.show()

**Remaining issues per source:**

In [ ]:
remaining = df_clean.groupby('source').agg(
    total=('source', 'size'),
    missing_price=('has_price', lambda x: (~x).sum()),
    missing_area=('has_area', lambda x: (~x).sum()),
    missing_address=('has_address', lambda x: (~x).sum()),
    avg_quality=('data_quality_score', 'mean'),
).round(2)
display(remaining)

## 14. Export / Handoff

Export dataset cho downstream pipeline:
- **Silver candidate**: `df_clean` (sau khi áp dụng versioned cleaning rules)
- **Gold candidate**: Cần thêm feature engineering & aggregation
- Format: Parquet (columnar, compressed)

In [ ]:
# Export path
export_dir = PROJECT_ROOT / "data" / "exports" / "eda"
export_dir.mkdir(parents=True, exist_ok=True)

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
export_path = export_dir / f"eda_clean_{timestamp}.parquet"

# Save
df_clean.to_parquet(export_path, index=False, engine='pyarrow')
print(f"✓ Exported: {export_path}")
print(f"  Shape: {df_clean.shape}")
print(f"  Size:  {export_path.stat().st_size / 1024:.1f} KB")

---

**Pipeline status:**
- ✅ Bronze → Latest-state → EDA profiling → Cleaning → Feature Engineering → Export
- ⏭ Next: Versioned cleaning rules → Silver dataset → Clean Analytical EDA → Gold